In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import shapiro, levene, mannwhitneyu

#importing the cleaned dataset
df = pd.read_csv("data_cleaned.csv")

# parse the date column to datetime format
df['date'] = pd.to_datetime(df['date'])


In [ ]:
# HYPOTHESIS TEST 1 : Does Daily Study Duration Differ Between Menstrual and Non-Menstrual Days?

# This code separates daily study hours into two groups—menstrual and non-menstrual—
# and prints the sample size for each group. These sample sizes are essential for
# evaluating statistical test assumptions and interpreting the analysis.

period_hours = df[df['is_period'] == 1]['study_hours_daily']
nonperiod_hours = df[df['is_period'] == 0]['study_hours_daily']

print("Period sample size:", len(period_hours))
print("Non-period sample size:", len(nonperiod_hours))


Period sample size: 43
Non-period sample size: 136


In [ ]:
# Shapiro–Wilk test evaluates whether each group follows a normal distribution.
# If p < 0.05, the distribution significantly deviates from normality.

print("Shapiro-Wilk Normality Test Results\n")

# Normality test for the period group
stat_p, p_p = shapiro(period_hours)

# Normality test for the non-period group
stat_np, p_np = shapiro(nonperiod_hours)

print(f"Period group → W={stat_p:.4f}, p-value={p_p:.5f}")
print(f"Non-period group → W={stat_np:.4f}, p-value={p_np:.5f}")

if p_p < 0.05:
    print("\nPeriod group is NOT normally distributed.")
else:
    print("\nPeriod group is normally distributed.")

if p_np < 0.05:
    print("Non-period group is NOT normally distributed.")
else:
    print("Non-period group is normally distributed.")


Shapiro-Wilk Normality Test Results

Period group → W=0.8387, p-value=0.00003
Non-period group → W=0.8627, p-value=0.00000

Period group is NOT normally distributed.
Non-period group is NOT normally distributed.


In [ ]:
# Levene’s test checks whether the variances of two groups are statistically equal.

stat_l, p_l = levene(period_hours, nonperiod_hours)

print("Levene Variance Equality Test")
print(f"Statistic={stat_l:.4f}, p-value={p_l:.5f}")

if p_l < 0.05:
    print("\nVariances are NOT equal.")
else:
    print("\nVariances are equal.")


Levene Variance Equality Test
Statistic=0.7669, p-value=0.38237

Variances are equal.


In [6]:
#Since both the normality and variance assumptions are violated,a parametric T-test is NOT appropriate.
#The correct statistical test is: Mann–Whitney U Test (non-parametric)
# Mann–Whitney U test compares the distributions of two independent groups.
# alternative="less" tests the hypothesis that: period_hours < nonperiod_hours

u_stat, p_val = mannwhitneyu(period_hours, nonperiod_hours, alternative='less')

print("Mann–Whitney U Test (Period < Non-period?)")
print(f"U statistic = {u_stat:.4f}")
print(f"p-value = {p_val:.6f}")

if p_val < 0.05:
    print("\nRESULT: Significant difference found!")
    print("We REJECT the null hypothesis (H₀).")
    print("Period days significantly REDUCE study hours.")
else:
    print("\nRESULT: No significant difference found.")
    print("We FAIL TO REJECT the null hypothesis.")
    print("Period days do NOT significantly reduce study hours.")


Mann–Whitney U Test (Period < Non-period?)
U statistic = 2888.0000
p-value = 0.450992

RESULT: No significant difference found.
We FAIL TO REJECT the null hypothesis.
Period days do NOT significantly reduce study hours.


In [7]:
# Effect size for Mann–Whitney U (Rank-Biserial Correlation)
# Interpretation:
# 0.1 = small effect, 0.3 = medium effect, 0.5 = large effect

n1 = len(period_hours)
n2 = len(nonperiod_hours)

rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
print("Effect Size (Rank-Biserial Correlation):", round(rank_biserial, 4))


Effect Size (Rank-Biserial Correlation): 0.0123


In [ ]:
# HYPOTHESIS TEST 2: Does Menstrual Period Increase Variability

#Computing descriptive variability measures (variance and standard deviation)for daily study hours during menstrual and non-menstrual periods
#to obtain whether study behavior is more unstable or dispersed during menstrual days before formal variance testing.

var_period = np.var(period_hours, ddof=1)
var_nonperiod = np.var(nonperiod_hours, ddof=1)

std_period = np.std(period_hours, ddof=1)
std_nonperiod = np.std(nonperiod_hours, ddof=1)

print("Variance and Standard Deviation")
print(f"Period days → Variance = {var_period:.4f}, Std = {std_period:.4f}")
print(f"Non-period days → Variance = {var_nonperiod:.4f}, Std = {std_nonperiod:.4f}")

Variance and Standard Deviation
Period days → Variance = 7.9648, Std = 2.8222
Non-period days → Variance = 6.9723, Std = 2.6405


In [ ]:
# Applying Levene’s test to evaluates whether the observed difference in variability is statistically significant
stat_l, p_l = levene(period_hours, nonperiod_hours)

print("Levene Variance Equality Test")
print(f"Statistic = {stat_l:.4f}")
print(f"p-value = {p_l:.6f}")

if p_l < 0.05:
    print("\nRESULT: Variances are significantly different.")
    print("Menstrual period may be associated with increased variability in study hours.")
else:
    print("\nRESULT: No significant difference in variances found.")
    print("Menstrual period does NOT significantly increase variability in study hours.")

Levene Variance Equality Test
Statistic = 0.7669
p-value = 0.382368

RESULT: No significant difference in variances found.
Menstrual period does NOT significantly increase variability in study hours.


In [ ]:
# HYPOTHESIS TEST 3: Do Study Hours Decrease on Days with Painkiller Usage?

#This test is to examine whether higher physical discomfort, which is highly realted with the painkiller usage, is associated with changes in study behavior.

#separating daily study hours into two groups based on painkiller usage status.
painkiller_yes = df[df['painkiller_usage'] == 1]['study_hours_daily']
painkiller_no = df[df['painkiller_usage'] == 0]['study_hours_daily']

print("Painkiller used days sample size:", len(painkiller_yes))
print("Painkiller not used days sample size:", len(painkiller_no))

Painkiller used days sample size: 58
Painkiller not used days sample size: 121


In [31]:
# Applying the Shapiro–Wilk normality test to daily study hours for days with and without painkiller usage.
from scipy.stats import shapiro, mannwhitneyu

print("Shapiro-Wilk Normality Test Results")

stat_y, p_y = shapiro(painkiller_yes)
stat_n, p_n = shapiro(painkiller_no)

print(f"Painkiller YES → W={stat_y:.4f}, p-value={p_y:.5f}")
print(f"Painkiller NO → W={stat_n:.4f}, p-value={p_n:.5f}")

Shapiro-Wilk Normality Test Results
Painkiller YES → W=0.9090, p-value=0.00036
Painkiller NO → W=0.8229, p-value=0.00000


In [32]:
# Since normality is violated, we apply Mann–Whitney U test
# The one-sided alternative hypothesis tests whether study duration is lower on days when painkillers are used, indicating a potential effect of pain severity on academic productivity.

u_stat, p_val = mannwhitneyu(
    painkiller_yes,
    painkiller_no,
    alternative='less'
)

print("Mann–Whitney U Test (Painkiller YES < NO?)")
print(f"U statistic = {u_stat:.4f}")
print(f"p-value = {p_val:.6f}")

if p_val < 0.05:
    print("\nRESULT: Significant difference found!")
    print("Painkiller usage is associated with reduced study hours.")
else:
    print("\nRESULT: No significant difference found.")
    print("Painkiller usage does not significantly reduce study hours.")

Mann–Whitney U Test (Painkiller YES < NO?)
U statistic = 4417.5000
p-value = 0.998003

RESULT: No significant difference found.
Painkiller usage does not significantly reduce study hours.


In [ ]:
# Effect size for Mann–Whitney U (Rank-Biserial Correlation)
# Interpretation:
# 0.1 = small effect, 0.3 = medium effect, 0.5 = large effect

n1 = len(painkiller_yes)
n2 = len(painkiller_no)

rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
print("Effect Size (Rank-Biserial Correlation):", round(rank_biserial, 4))

Effect Size (Rank-Biserial Correlation): -0.2589


In [ ]:
# HYPOTHESİS TEST 4: Does Exam Period Masks Menstrual Effect?

# Seperating non-exam days and further separates them into menstrual and non-menstrual groups to examine whether menstrual effects on study duration are more observable in the absence of exam-related academic pressure.

print("Analysis for NON-EXAM DAYS (exam_period = 0)\n")

df_no_exam = df[df['exam_period'] == 0]
period_no_exam = df_no_exam[df_no_exam['is_period'] == 1]['study_hours_daily']
nonperiod_no_exam = df_no_exam[df_no_exam['is_period'] == 0]['study_hours_daily']

print("Period days (no exam) sample size:", len(period_no_exam))
print("Non-period days (no exam) sample size:", len(nonperiod_no_exam))

Analysis for NON-EXAM DAYS (exam_period = 0)

Period days (no exam) sample size: 33
Non-period days (no exam) sample size: 108


In [ ]:
# Normality check

stat_p_ne, p_p_ne = shapiro(period_no_exam)
stat_np_ne, p_np_ne = shapiro(nonperiod_no_exam)

print("Shapiro-Wilk Normality Test (No Exam)")
print(f"Period → W={stat_p_ne:.4f}, p-value={p_p_ne:.5f}")
print(f"Non-period → W={stat_np_ne:.4f}, p-value={p_np_ne:.5f}")

Shapiro-Wilk Normality Test (No Exam)
Period → W=0.7558, p-value=0.00001
Non-period → W=0.8315, p-value=0.00000


In [ ]:
# Since normality is violated, we apply Mann–Whitney U test

u_ne, p_ne = mannwhitneyu(
    period_no_exam,
    nonperiod_no_exam,
    alternative='less'
)

print("Mann–Whitney U Test (No Exam | Period < Non-period?)")
print(f"U statistic = {u_ne:.4f}")
print(f"p-value = {p_ne:.6f}")

if p_ne < 0.05:
    print("\nRESULT: Significant menstrual effect during NON-exam days.")
else:
    print("\nRESULT: No significant menstrual effect during NON-exam days.")

Mann–Whitney U Test (No Exam | Period < Non-period?)
U statistic = 1679.5000
p-value = 0.299496

RESULT: No significant menstrual effect during NON-exam days.


In [ ]:
# Effect size for Mann–Whitney U (Rank-Biserial Correlation)
# Interpretation:
# 0.1 = small effect, 0.3 = medium effect, 0.5 = large effect

n1_ne = len(period_no_exam)
n2_ne = len(nonperiod_no_exam)
rank_biserial_ne = 1 - (2 * u_ne) / (n1_ne * n2_ne)
print("Effect Size (Rank-Biserial Correlation):", round(rank_biserial_ne, 4))

Effect Size (Rank-Biserial Correlation): 0.0575


In [ ]:
# Seperating exam days and separates them into menstrual and non-menstrual groups to evaluate whether exam-related academic pressure alters or masks the potential effect of menstrual status on daily study duration.

print("Analysis for EXAM DAYS (exam_period = 1)\n")

df_exam = df[df['exam_period'] == 1]

period_exam = df_exam[df_exam['is_period'] == 1]['study_hours_daily']
nonperiod_exam = df_exam[df_exam['is_period'] == 0]['study_hours_daily']

print("Period days (exam) sample size:", len(period_exam))
print("Non-period days (exam) sample size:", len(nonperiod_exam))

Analysis for EXAM DAYS (exam_period = 1)

Period days (exam) sample size: 10
Non-period days (exam) sample size: 28


In [ ]:
# Normality check

stat_p_e, p_p_e = shapiro(period_exam)
stat_np_e, p_np_e = shapiro(nonperiod_exam)

print("Shapiro-Wilk Normality Test (Exam)")
print(f"Period → W={stat_p_e:.4f}, p-value={p_p_e:.5f}")
print(f"Non-period → W={stat_np_e:.4f}, p-value={p_np_e:.5f}")

Shapiro-Wilk Normality Test (Exam)
Period → W=0.9459, p-value=0.61981
Non-period → W=0.8988, p-value=0.01071


In [ ]:
# Since normality is violated, we apply Mann–Whitney U test

u_e, p_e = mannwhitneyu(
    period_exam,
    nonperiod_exam,
    alternative='less'
)

print("Mann–Whitney U Test (Exam | Period < Non-period?)")
print(f"U statistic = {u_e:.4f}")
print(f"p-value = {p_e:.6f}")

if p_e < 0.05:
    print("\nRESULT: Significant menstrual effect during EXAM days.")
else:
    print("\nRESULT: No significant menstrual effect during EXAM days.")

Mann–Whitney U Test (Exam | Period < Non-period?)
U statistic = 133.5000
p-value = 0.421141

RESULT: No significant menstrual effect during EXAM days.


In [ ]:
# Effect size for Mann–Whitney U (Rank-Biserial Correlation)
# Interpretation:
# 0.1 = small effect, 0.3 = medium effect, 0.5 = large effect

n1_e = len(period_exam)
n2_e = len(nonperiod_exam)
rank_biserial_e = 1 - (2 * u_e) / (n1_e * n2_e)
print("Effect Size (Rank-Biserial Correlation):", round(rank_biserial_e, 4))

Effect Size (Rank-Biserial Correlation): 0.0464
